By Mohammad Reza Nilchiyan

## Load data and merge database

In [42]:
import pandas as pd
# Temperature dataset
tx_path = "/Users/mohammadrezanilchiyan/Desktop/UE/Datascientest/project/Temperature/Data/processed_2025.12.15/Marseille_daily_TX_raw.csv"

# Wind dataset
wind_path = "/Users/mohammadrezanilchiyan/Desktop/UE/Datascientest/project/Temperature/Data/processed_2025.12.15/temp_wind_marseille.csv"


In [43]:
# Temperature
df_tx = pd.read_csv(tx_path, parse_dates=["date"])

# Wind
df_wind = pd.read_csv(wind_path, parse_dates=["date"])

print("df_tx shape:", df_tx.shape)
print("df_wind shape:", df_wind.shape)
print("Unique dates in df_tx:", len(df_tx['date'].unique()))
print("Unique dates in df_wind:", len(df_wind['date'].unique()))

df_tx shape: (133133, 8)
df_wind shape: (133133, 11)
Unique dates in df_tx: 27028
Unique dates in df_wind: 27028


In [44]:
# Temperature dataset: keep Station name (NOM_USUEL) and TX
df_tx = df_tx[["date", "TX", "NOM_USUEL"]]

# Wind dataset: keep only rows where main features exist
df_wind = df_wind.dropna(subset=["wind_max_inst_ms", "temp_max_c", "wind_dir_inst_deg", "wind_mean_10m_ms"])

# Rename columns for clarity
df_wind = df_wind.rename(
    columns={
        "wind_max_inst_ms": "Wx",     # max instantaneous wind
        "temp_max_c": "Tx",           # max temperature
        "wind_dir_inst_deg": "Wx_dir", # wind direction
        "wind_mean_10m_ms" : "Wm_10"
    
    }
)


In [45]:
df_merged = df_tx.merge(df_wind, on="date", how="inner").sort_values("date").reset_index(drop=True)

In [46]:
#df_merged["Wm_10"] = df_merged["Wm_10"].fillna(df_merged["Wm_10"].median())


##  Predictor features preparation

In [47]:
# wind direction encoding
import numpy as np

df_merged["Wx_dir_rad"] = np.deg2rad(df_merged["Wx_dir"])
df_merged["Wx_dir_sin"] = np.sin(df_merged["Wx_dir_rad"])
df_merged["Wx_dir_cos"] = np.cos(df_merged["Wx_dir_rad"])

selected_cols = ["date", "NOM_USUEL", "TX", "Tx", "Wx", "Wx_dir_sin", "Wx_dir_cos", "Wm_10"]
df_selected = df_merged[selected_cols]
df_selected_clean = df_selected.dropna(subset=selected_cols)

In [48]:
df_selected_clean.head(7)

,date,NOM_USUEL,TX,Tx,Wx,Wx_dir_sin,Wx_dir_cos,Wm_10
0,1950-01-01,MARIGNANE,11.4,11.4,4.0,-0.939693,0.342020,0.3
1,1950-01-01,ISTRES,10.5,11.4,4.0,-0.939693,0.342020,0.3
2,1950-01-01,SALON DE PROVENCE,9.8,11.4,4.0,-0.939693,0.342020,0.3
3,1950-01-01,BEC DE L AIGLE,13.6,11.4,4.0,-0.939693,0.342020,0.3
4,1950-01-02,MARIGNANE,9.0,9.0,9.0,-0.642788,0.766044,2.1
5,1950-01-02,SALON DE PROVENCE,8.0,9.0,9.0,-0.642788,0.766044,2.1
6,1950-01-02,BEC DE L AIGLE,12.0,9.0,9.0,-0.642788,0.766044,2.1


In [49]:
print("Merged DataFrame shape:", df_selected_clean.shape)

Merged DataFrame shape: (480518, 8)


# Modeling 

In [51]:
#checks, for each day, Is the daily maximum temperature (TX) at least 35°C? , by end
#Convert True / False to numbers. True (1) and False(0)
df_selected_clean["extreme_heat"] = (df_selected_clean['TX'] >= 35).astype(int)

/var/folders/c1/08tt9xg55rj5b78fpdm6c7n00000gn/T/ipykernel_26466/2996790761.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_selected_clean["extreme_heat"] = (df_selected_clean['TX'] >= 35).astype(int)


# Oversample minority class using SMOTE & Gradient Boosting Classifier

- Handle imbalance:Oversample extreme heat days (SMOTE, RandomOverSampler)

- Try more powerful classifiers:
Random Forest, Gradient Boosting (XGBoost) handle imbalanced classes better.

In [52]:
features = ["Wx", "Wx_dir_sin", "Wx_dir_cos", "Wm_10"]
X = df_selected_clean[features]
y = df_selected_clean["extreme_heat"]

## Train / test split

In [55]:

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y, # VERY important for imbalance # every class is guaranteed to appear in both sets.
    random_state=42
)


## Handle imbalance (SMOTE)

In [56]:
# Step 4: Oversampling on training set
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)


## Train Gradient Boosting classifier on oversampled data

In [58]:
# Step 5: Train Gradient Boosting
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

gb.fit(X_train_res, y_train_res)


,loss,'log_loss'
,learning_rate,0.05
,n_estimators,300
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,3
,min_impurity_decrease,0.0
,init,None


### Predict on the test set

In [59]:
# Step 6
#didn’t oversample on the test set
y_pred = gb.predict(X_test)

In [60]:
from sklearn.metrics import accuracy_score

acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc:.4f}")

Accuracy: 0.7942


In [61]:
import numpy as np

unique, counts = np.unique(y_test, return_counts=True)
print(dict(zip(unique, counts)))

{np.int64(0): np.int64(95421), np.int64(1): np.int64(683)}


In [62]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
print(cm)

[[75882 19539]
 [  236   447]]


In [63]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      0.80      0.88     95421
           1       0.02      0.65      0.04       683

    accuracy                           0.79     96104
   macro avg       0.51      0.72      0.46     96104
weighted avg       0.99      0.79      0.88     96104

